# Fee Concentration Duration Model — Results

**Spec:** `specs/econometrics/lp-insurance-demand.tex`

**Model:** log(blocklife) = β₀ + β₁·max_A_T + β₂·IL + ε (OLS + HC1 robust SEs)

**Hypothesis:** β₁ < 0 → higher fee concentration shortens position life → insurance demand

In [ ]:
import sys
sys.path.insert(0, "..")

from econometrics.data import DAILY_AT_MAP, IL_MAP, RAW_POSITIONS
from econometrics.ingest import build_lagged_positions
from econometrics.duration import (
    duration_model_robust,
    economic_magnitude,
    nested_models,
    quartile_analysis,
    sensitivity_sweep,
)

LAG = 5
positions = build_lagged_positions(RAW_POSITIONS, DAILY_AT_MAP, IL_MAP, lag_days=LAG)
print(f"Positions after lag={LAG} filter: {len(positions)}")
print(f"Date range: {positions[0].burn_date} to {positions[-1].burn_date}")

## 1. Main Result

In [ ]:
result = duration_model_robust(positions, measure="max")
print(f"{'':>20} {'Coeff':>10} {'OLS SE':>10} {'HC1 SE':>10} {'HC1 p':>10}")
print(f"{'\u03b2\u2081 (max A_T)':>20} {result.beta_a_t:>10.4f} {result.se_a_t:>10.4f} {result.robust_se_a_t:>10.4f} {result.robust_p_value_a_t:>10.6f}")
print(f"{'\u03b2\u2082 (IL)':>20} {result.beta_il:>10.4f} {result.se_il:>10.4f} {result.robust_se_il:>10.4f} {result.robust_p_value_il:>10.6f}")
print(f"{'\u03b2\u2080 (intercept)':>20} {result.beta_intercept:>10.4f}")
print(f"\nn={result.n_obs}  R\u00b2={result.r_squared:.4f}  Mean blocklife={result.mean_blocklife_hours:.1f} hours")

## 2. Economic Magnitude

In [ ]:
mag = economic_magnitude(result, delta_a_t=0.10)
print(f"A 0.10 increase in max A_T:")
print(f"  Multiplicative factor: {mag['factor']:.4f}")
print(f"  Position life change:  {mag['pct_change']:+.1f}%")
print(f"  Hours shortened:       {mag['hours_shortened']:.1f} hours")
print(f"  (Mean position life:   {mag['mean_blocklife_hours']:.1f} hours)")

## 3. Dose-Response (Quartile Analysis)

In [ ]:
import matplotlib.pyplot as plt

quartiles = quartile_analysis(positions, measure="max")
qs = [q.quartile for q in quartiles]
hours = [q.mean_blocklife_hours for q in quartiles]
ats = [q.mean_a_t for q in quartiles]

fig, ax1 = plt.subplots(figsize=(8, 5))
bars = ax1.bar(qs, hours, color=["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"], edgecolor="black")
ax1.set_xlabel("max A_T Quartile")
ax1.set_ylabel("Mean Position Life (hours)")
ax1.set_title("Dose-Response: Fee Concentration vs Position Duration")
ax1.set_xticks(qs)
ax1.set_xticklabels([f"Q{q}\n(A_T={a:.3f})" for q, a in zip(qs, ats)])

for bar, h, n in zip(bars, hours, [q.n_obs for q in quartiles]):
    ax1.text(bar.get_x() + bar.get_width()/2, h + 2, f"{h:.0f}h\n(n={n})",
             ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Lag Sensitivity

In [ ]:
rows = sensitivity_sweep(RAW_POSITIONS, DAILY_AT_MAP, IL_MAP)
print(f"{'Lag':>4} {'Measure':>8} {'\u03b2\u2081':>10} {'HC1 SE':>10} {'p-value':>10} {'n':>6} {'Sig':>4}")
print("-" * 60)
for r in rows:
    sig = "***" if r.robust_p_value_a_t < 0.01 else "**" if r.robust_p_value_a_t < 0.05 else "*" if r.robust_p_value_a_t < 0.10 else ""
    print(f"{r.lag:>4} {r.measure:>8} {r.beta_a_t:>10.4f} {r.robust_se_a_t:>10.4f} {r.robust_p_value_a_t:>10.4f} {r.n_obs:>6} {sig:>4}")

## 5. Nested Model Comparison (IL Orthogonality)

In [ ]:
models = nested_models(positions, measure="max")
print(f"{'Model':>12} {'\u03b2\u2081(A_T)':>10} {'HC1 p':>10} {'\u03b2\u2082(IL)':>10} {'HC1 p':>10} {'R\u00b2':>8}")
print("-" * 65)
for name, m in models.items():
    print(f"{name:>12} {m.beta_a_t:>10.4f} {m.robust_p_value_a_t:>10.4f} {m.beta_il:>10.4f} {m.robust_p_value_il:>10.4f} {m.r_squared:>8.4f}")

## 6. Implied Insurance Value

In [ ]:
import math

FEE_REVENUE_PER_HOUR = 100.0  # placeholder: $100/hr avg fee revenue per LP

if result.beta_a_t < 0:
    mean_at = sum(p.max_a_t for p in positions) / len(positions)
    hours_lost = abs(mag["hours_shortened"])
    implied_value = hours_lost * FEE_REVENUE_PER_HOUR
    print(f"Mean max A_T exposure:    {mean_at:.4f}")
    print(f"Hours shortened per 0.10: {hours_lost:.1f}")
    print(f"Fee revenue per hour:     ${FEE_REVENUE_PER_HOUR:.0f} (placeholder)")
    print(f"Implied insurance value:  ${implied_value:.0f} per position per 0.10 A_T shock")
    print(f"\nINTERPRETATION: PLPs would pay up to ${implied_value:.0f} to avoid")
    print(f"a 0.10 increase in fee concentration. This validates insurance demand.")
else:
    print("\u03b2\u2081 \u2265 0 \u2014 concentration does not shorten positions in this sample.")
    print("Insurance demand signal not found with current specification.")